---
title: Глава 10. Соединения
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-09-25
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Инструмент сборки статических сайтов
    JupySQL: Расширение для запуска и подсветки SQL в Jupyter
    GitHub: Платформа хостинга репозиториев и совместной разработки
    GitHub Pages: Сервис бесплатного хостинга статических сайтов
    GitHub Actions: Платформа автоматизации рабочих процессов и CI/CD
    Pandas: Библиотека Python для анализа и обработки данных
    Polars: Мощный аналог Pandas на Rust/Python
---

In [1]:
import pandas as pd
import sql
import sqlalchemy as sa

pd.set_option('display.max_rows', 20)

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.autopandas = True

connection_url = sa.engine.URL.create(
    drivername='mysql+pymysql',
    host='localhost',
    port=3306,
    database='sakila',
    username='root',
    password='********',
)
engine = sa.create_engine(connection_url)

%sql engine

print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 20")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")

Pandas ver. 3.0.5: порог усечения строк уменьшен до 20
SQLAlchemy ver. 2.0.52: подключение создано
JupySQL ver. 0.11.1: подключен через SQLAlchemy Engine


К настоящему времени вы уже должны быть хорошо знакомы с концепцией *врутреннего* соединения, которая была представлена в главе 5.

:::{figure} ./media/inner-join.png
:alt: INNER JOIN
:align: center
Внутреннее соединение **INNER JOIN**
:::

В этой главе рассматриваются другие способы соединения таблиц, включая _**внешнее**_ соединение и _**перекрестное**_ соединение.

## Внешние соединения

До сих пор во всех примерах, включающих несколько таблиц, нас не волновало, что условия соединения могут не найти совпадений для всех строк таблицы.

Например, таблица **inventory** содержит строку для каждого фильма, доступного для проката, но из 1000 строк в таблице **film** только 958 имеют одну или несколько строк в таблице **inventory**. Остальные 42 фильма для проката недоступны (возможно это новинки, которые должны прибыть в пункты проката со дня на день), поэтому идентификаторы этих фильмов в таблице **inventory** отсутствуют.

Следующий запрос подсчитывает количество доступных копий каждого фильма с помощью соединения этих двух таблиц:

In [3]:
%%sql
SELECT
    f.film_id,
    f.title,
    COUNT(*) AS num_copies
FROM film f
    INNER JOIN inventory i
        ON f.film_id = i.film_id
GROUP BY
    f.film_id,
    f.title;

958 rows affected.

,film_id,title,num_copies
0,1,ACADEMY DINOSAUR,8
1,2,ACE GOLDFINGER,3
2,3,ADAPTATION HOLES,4
3,4,AFFAIR PREJUDICE,7
4,5,AFRICAN EGG,3
...,...,...,...
953,996,YOUNG LANGUAGE,2
954,997,YOUTH KICK,2
955,998,ZHIVAGO CORE,2
956,999,ZOOLANDER FICTION,5


Хотя можно было бы ожидать, что будет возвращено 1000 строк (по одной для каждого фильма), запрос возвращает только 958 строк.

Дело в том, что запрос использует *внутреннее* соединение, которое возвращает только строки, удовлетворяющие условию соединения. Например, фильм *Alice Fantasia* (**film_id** равен 14) не отображается в результатах, потому что для него нет строк в таблице **inventory**.

Если хотите, чтобы запрос возвращал все 1000 фильмов, независимо от того, имеются ли соответствующие строки в таблице **inventory**, можете использовать *внешнее* соединение, которое, по сути, делает условие соединения необязательным:

In [9]:
%%sql
SELECT
    f.film_id,
    f.title,
    COUNT(i.inventory_id) AS num_copies
FROM film f
    LEFT OUTER JOIN inventory i
        ON f.film_id = i.film_id
GROUP BY
    f.film_id,
    f.title
ORDER BY num_copies;

1000 rows affected.

,film_id,title,num_copies
0,14,ALICE FANTASIA,0
1,33,APOLLO TEEN,0
2,36,ARGONAUTS TOWN,0
3,38,ARK RIDGEMONT,0
4,41,ARSENIC INDEPENDENCE,0
...,...,...,...
995,897,TORQUE BOUND,8
996,911,TRIP NEWTON,8
997,945,VIRGINIAN PLUTO,8
998,973,WIFE TURN,8


Как видите, запрос теперь возвращает все 1000 строк из таблицы **film**, при этом 42 строки из общего количества строк (включая фильм *Alice Fantasia*) имеют значение 0 в столбце **num_copies**, что указывает на отсутствие доступных для проката копий.